In [1]:
import os

region = os.environ.get("NEBIUS_REGION")
registry_path = os.environ.get("REGISTRY_PATH")

print("Region:", region)
print("Registry:", registry_path)


In [2]:
from dask_kubernetes.operator import KubeCluster, make_cluster_spec
from dask.distributed import Client


config = {
    "name": "mk8s-dask-cluster",
    "image": f"cr.{region}.nebius.cloud/{registry_path}/mda-dask:latest",
    "n_workers": 7,
    "resources":{"requests": {"memory": "2Gi", "cpu": "800m"}, 
                 "limits": {"memory": "4Gi", "cpu": "1"}}
}

cluster = KubeCluster(**config)
client = Client(cluster)
client


In [3]:
def check_import():
    import MDAnalysis
    return MDAnalysis.__version__

client.submit(check_import).result()


In [4]:
import dask
import MDAnalysis as mda
from dask.distributed import Client
from MDAnalysis.analysis.backends import BackendBase
from MDAnalysis.analysis.dssp import DSSP
from dask.distributed import Client
from dask_kubernetes.operator import KubeCluster

class DistributedBackend(BackendBase):
    def __init__(self, n_workers):
        super().__init__(n_workers)

    def assign_client(self, client):
        self.client = client

    def apply(self, func, computations):
        return self.client.compute([dask.delayed(func)(c) for c in computations], sync=True)

u = mda.Universe("YiiP_lipids.gro.gz", "YiiP_lipids.xtc", in_memory=True)

backend = DistributedBackend(n_workers=5)
backend.assign_client(client)

# test computation
tasks = [dask.delayed(lambda x: x + 1)(i) for i in range(11)]
print(f"{client.compute(tasks, sync=True)=}")

print(DSSP(u).run(backend=backend, unsupported_backend=True).results)


In [5]:
import time
import dask
from dask.distributed import wait

def slow_increment(x, delay=0.5):
    time.sleep(delay)
    return x + 1

def benchmark_workers(client, n_tasks=100, delay=0.5, worker_counts=(1, 2, 4, 6)):
    results = {}

    for n_workers in worker_counts:
        print(f"\n=== n_workers = {n_workers} ===")
        # масштабируем кластер
        client.cluster.scale(n_workers)
        client.wait_for_workers(n_workers)

        # создаём задачи
        tasks = [dask.delayed(slow_increment)(i, delay=delay) for i in range(n_tasks)]

        # запускаем и меряем время
        t0 = time.perf_counter()
        futures = client.compute(tasks)
        _ = client.gather(futures)
        t1 = time.perf_counter()

        elapsed = t1 - t0
        results[n_workers] = elapsed
        print(f"Total time: {elapsed:.2f} s")

    return results

results = benchmark_workers(client, n_tasks=21, delay=0.5, 
                            worker_counts=(1, 2, 3, 4, 5, 6, 7))
print("\nSummary:")
for n, t in results.items():
    print(f"{n} workers: {t:.2f} s")


In [6]:
# cluster.scale(5)
# cluster.close()
